# **1. Data Understanding**

In [ ]:
import pandas as pd

df = pd.read_csv("/content/sample_data/IMDB Dataset.csv")

print("Shape:", df.shape)

print("\nClass Distribution:\n", df['sentiment'].value_counts())

print("\nSample Texts:")
print(df.head())

Shape: (50000, 2)

Class Distribution:
 sentiment
positive    25000
negative    25000
Name: count, dtype: int64

Sample Texts:
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [ ]:
!pip install nltk

In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

# **2. NLP Preprocessing**

In [18]:
import re
import string
import nltk

# Download required data
nltk.download('stopwords')
nltk.download('punkt')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

# Initialize tools
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

# Cleaning function
def clean_text(text):
    text = text.lower()  # Lowercase
    text = re.sub(r"http\S+|www\S+", "", text)  # Remove URLs
    text = re.sub(r'\d+', '', text)  # Remove numbers
    text = text.translate(str.maketrans('', '', string.punctuation))  # Remove punctuation
    return text

# Preprocessing function
def preprocess_text(text):
    text = clean_text(text)
    tokens = word_tokenize(text)  # Tokenization
    tokens = [word for word in tokens if word not in stop_words]  # Remove stopwords
    tokens = [stemmer.stem(word) for word in tokens]  # Stemming
    return " ".join(tokens)

# Apply preprocessing
df['clean_text'] = df['review'].apply(preprocess_text)

print(df[['review', 'clean_text']].head())

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


                                              review  \
0  One of the other reviewers has mentioned that ...   
1  A wonderful little production. <br /><br />The...   
2  I thought this was a wonderful way to spend ti...   
3  Basically there's a family where a little boy ...   
4  Petter Mattei's "Love in the Time of Money" is...   

                                          clean_text  
0  one review mention watch oz episod youll hook ...  
1  wonder littl product br br film techniqu unass...  
2  thought wonder way spend time hot summer weeke...  
3  basic there famili littl boy jake think there ...  
4  petter mattei love time money visual stun film...  


# **3. Feature Engineering**

In [20]:
# Bag of Words

from sklearn.feature_extraction.text import CountVectorizer

bow = CountVectorizer(max_features=5000)
X_bow = bow.fit_transform(df['clean_text']).toarray()

In [21]:
# TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['clean_text']).toarray()


# **4. Model Building**


In [23]:
from sklearn.model_selection import train_test_split

# Target variable
y = df['sentiment']

# Train-test split (using TF-IDF)
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)

In [24]:
# Logistic Regression

from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()
lr.fit(X_train, y_train)

LogisticRegression()

In [25]:
# Naive Bayes

from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train, y_train)

MultinomialNB()

In [26]:
# Decision Tree
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)


DecisionTreeClassifier()

# **5. Model Evaluation**


In [27]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)

    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average='weighted'),
        "Recall": recall_score(y_test, y_pred, average='weighted'),
        "F1 Score": f1_score(y_test, y_pred, average='weighted')
    }

# Evaluate all models
results = {
    "Logistic Regression": evaluate(lr, X_test, y_test),
    "Naive Bayes": evaluate(nb, X_test, y_test),
    "Decision Tree": evaluate(dt, X_test, y_test)
}

print(results)

{'Logistic Regression': {'Accuracy': 0.8854, 'Precision': 0.8857385788209, 'Recall': 0.8854, 'F1 Score': 0.8853590336622783}, 'Naive Bayes': {'Accuracy': 0.846, 'Precision': 0.8460413308191465, 'Recall': 0.846, 'F1 Score': 0.8459860016573743}, 'Decision Tree': {'Accuracy': 0.715, 'Precision': 0.7150338436121838, 'Recall': 0.715, 'F1 Score': 0.715004309255848}}


# **6. Comparison & Insights**


> **Best Preprocessing Steps**



* **Lowercasing** → avoids duplicates (“Good” vs “good”)

* **Remove punctuation & special characters** → reduces noise

* **Remove stopwords** → keeps only meaningful words

* **Tokenization** → splits text into words

* **Stemming / Lemmatization** → converts words to root form *(“running → run”)*

* **Remove URLs & numbers** → not useful for sentiment


> **Best Vectorization
Technique :** TF-IDF

**Why TF-IDF is better than Bag of Words:**

* Considers importance of words, not just frequency.
* Reduces impact of common words.
* Works very well for classification tasks


> **Best Model :** Logistic Regression

**Why it performs best:**

* Handles high-dimensional sparse data well.
* Works great with TF-IDF.
* Stable and less prone to overfitting.






> **Trade-offs**


**Accuracy vs Speed**

* Naive Bayes -> Fast but slightly less accurate

* Logistic Regression -> Accurate but slower

**Simplicity vs Performance**

* Bag of Words -> Simple but weaker

* TF-IDF -> Slightly complex but better

**Interpretability vs Power**

* Decision Tree -> Easy to explain
* Logistic Regression -> Better performance

